# Predictive Analytics: Neural Network - Advanced Baseline v2

Note: theoretically we would need to either encode the spatial feature (i. e. community area) or train a separate model for each spatial unit. Otherwise the feature "community_area" is a continuous numeric feature!

Like v1 but with model architecture from baseline to check if log1p is helping or not

=> v2 is better than v1 and baseline -> log1p should be helpful

In [1]:
import polars as pl
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import math
import matplotlib.pyplot as plt

from run_config import MODELS_DIR, PATHS

from helper_functions import (
    get_torch_device,
    load_model_from_checkpoint,
    collect_predictions,
    compute_regression_metrics,
    plot_regression_diagnostics,
)

## Preparations

In [2]:
# Dataset loop configuration
DATASET_CONFIGS = (
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "1h"},
    {"spatial_unit": "census_tracts", "spatial_col": "census_tract", "time_unit": "1h"},
    {"spatial_unit": "community_areas", "spatial_col": "community_area", "time_unit": "1h"},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "2h"},
    {"spatial_unit": "census_tracts", "spatial_col": "census_tract", "time_unit": "2h"},
    {"spatial_unit": "community_areas", "spatial_col": "community_area", "time_unit": "2h"},
    {"spatial_unit": "hexagon", "spatial_col": "h3_cell", "time_unit": "4h"},
    {"spatial_unit": "census_tracts", "spatial_col": "census_tract", "time_unit": "4h"},
    {"spatial_unit": "community_areas", "spatial_col": "community_area", "time_unit": "4h"},
)
MODEL_PATH = MODELS_DIR
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
]

        

### Load the data and select features and target

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

Impotant, check datatype of features and target. Everything should be numerical. We already applied encoding to categorical features (e. g. cloud coverage), therefore all features are numerical.

### Scale the train data

Important, the scaler is only fitted on X_train to prevent data leakage

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

### PyTorch Dataset and DataLoader

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

**BATCH_SIZE** we start with a standard batch_size of 1024 as we have a lot of training data and a smaller batch_size would lead to too many updates in each epoche

Possible, other used batch_sizes could be: 32, 64, 128, 256, 512, 1024, 2048

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

## Advanced Baseline v2

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

### Model architecture


In [8]:
class TaxiDemandAdvancedNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

There it important to check that the shape is the batch_size and the right dimension, i. e. number of features / target

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

### Loss function and optimizer

As loss we start with MSELoss, which means that larger errors are larger penalized. And we use ADAM as optimizer with a learning rate of 0.001 as of now.

Options for further models:
- L1Loss(): more similar to MAE

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

### Trainingsloop and Validation

In [14]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    
    total_loss = 0.0
    total_samples = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # 1. Alte Gradienten löschen
        optimizer.zero_grad()
        
        # 2. Forward Pass
        y_pred = model(X_batch)
        
        # 3. Loss berechnen
        loss = criterion(y_pred, y_batch)
        
        # 4. Backpropagation
        loss.backward()
        
        # 5. Gewichte updaten
        optimizer.step()
        
        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
    
    avg_loss = total_loss / total_samples
    return avg_loss

def evaluate(model, val_loader, criterion, device):
    model.eval()
    
    total_loss = 0.0
    total_absolute_error = 0.0
    total_squared_error = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(X_batch)

            y_batch = y_batch.view_as(y_pred)

            loss = criterion(y_pred, y_batch)
            
            # back transformation
            y_pred_original = torch.expm1(y_pred)
            y_true_original = torch.expm1(y_batch)

            y_pred_original = torch.clamp(y_pred_original, min=0)
            
            batch_size = X_batch.size(0)
            
            total_loss += loss.item() * batch_size

            # MAE/RMSE on original scaled data
            total_absolute_error += torch.sum(
                torch.abs(y_pred_original - y_true_original)
            ).item()

            total_squared_error += torch.sum(
                (y_pred_original - y_true_original) ** 2
            ).item()

            total_samples += batch_size
    
    avg_loss = total_loss / total_samples
    mae = total_absolute_error / total_samples
    rmse = (total_squared_error / total_samples) ** 0.5
    
    return avg_loss, mae, rmse

### Start trainingsloop

We use checkpoint to store the models parameter after each epoche

In [15]:
def train_dataset_v2(dataset_config):
    SPATIAL_UNIT = dataset_config["spatial_unit"]
    SPATIAL_COL = dataset_config["spatial_col"]
    TIME_UNIT = dataset_config["time_unit"]
    DATASET_STEM = f"GOLD_{TIME_UNIT.upper()}_DEMAND_{SPATIAL_UNIT.upper()}"
    MODEL_TAG = f"{SPATIAL_UNIT}_{TIME_UNIT}"
    TRAINING_DATA_PATH = PATHS.train_test_dir / f"{DATASET_STEM}_TRAIN.parquet"
    VAL_DATA_PATH = PATHS.train_test_dir / f"{DATASET_STEM}_VAL.parquet"

    print(f"\n=== Training v2: {MODEL_TAG} ===")
    print(f"Train: {TRAINING_DATA_PATH}")
    print(f"Validation: {VAL_DATA_PATH}")
    # Load the tranining data
    train_df = pl.scan_parquet(TRAINING_DATA_PATH)
    val_df = pl.scan_parquet(VAL_DATA_PATH)

    # Create X and Y data
    feature_cols = [
        col for col in train_df.collect_schema().names()
        if col not in EXCLUDE_COLS
    ]

    X_train = train_df.select(feature_cols).collect()
    y_train = train_df.select(TARGET_COL).collect()

    X_val = val_df.select(feature_cols).collect()
    y_val = val_df.select(TARGET_COL).collect()

    print("Features: ", X_train.schema)
    print("Target: ", y_train.schema)
    if X_train.height == 0 or X_val.height == 0:
        raise ValueError(f"Empty train or validation split for {MODEL_TAG}")

    # Encode the spatial identifier consistently for train and validation.
    spatial_values = sorted(
        X_train.select(pl.col(SPATIAL_COL).cast(pl.String)).to_series().unique().to_list()
    )
    spatial_to_idx = {value: index for index, value in enumerate(spatial_values)}
    X_train = X_train.with_columns(
        pl.col(SPATIAL_COL)
        .cast(pl.String)
        .replace_strict(spatial_to_idx, default=-1)
        .cast(pl.Float32)
        .alias(SPATIAL_COL)
    )
    X_val = X_val.with_columns(
        pl.col(SPATIAL_COL)
        .cast(pl.String)
        .replace_strict(spatial_to_idx, default=-1)
        .cast(pl.Float32)
        .alias(SPATIAL_COL)
    )

    # transform to numpy as it works best with scaler
    X_train_np = X_train.to_numpy()
    X_val_np = X_val.to_numpy()

    # fit scaler
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_np) # fit only called on train
    X_val_scaled = scaler.transform(X_val_np) # only transformed, not fitted!

    # prepare target and features
    # log1p transformation
    y_train_np = np.log1p(y_train.to_numpy()).astype("float32")
    y_val_np = np.log1p(y_val.to_numpy()).astype("float32")
    X_train_scaled = X_train_scaled.astype("float32")
    X_val_scaled = X_val_scaled.astype("float32")

    print(X_train_scaled.shape)
    print(y_train_np.shape)

    print(X_val_scaled.shape)
    print(y_val_np.shape)
    # Convert to pytorch tensor
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32)

    X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val_np, dtype=torch.float32)

    print(X_train_tensor.shape)
    print(y_train_tensor.shape)

    print(X_val_tensor.shape)
    print(y_val_tensor.shape)

    # Tensor dataset
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    # Create the data loader
    BATCH_SIZE = 1024

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # Test
    X_batch, y_batch = next(iter(train_loader))

    print(X_batch.shape)
    print(y_batch.shape)
    # model needs the number of features
    input_dim = X_train_tensor.shape[1]

    print("Input Features:", input_dim)
    # initialise model
    model = TaxiDemandAdvancedNN(
        input_dim=input_dim,
    )
    print(model)
    # Test with one batch
    X_batch, y_batch = next(iter(train_loader))

    y_pred = model(X_batch)

    print("Input batch:", X_batch.shape)
    print("Prediction batch:", y_pred.shape)
    print("Target batch:", y_batch.shape)
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print("Using device:", device)
    model = model.to(device)
    criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    EPOCHS = 100
    PATIENCE = 10
    MIN_DELTA = 0


    train_losses = []
    val_losses = []
    val_maes = []
    val_rmses = []


    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(
            model=model,
            train_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device
        )
    
        val_loss, val_mae, val_rmse = evaluate(
            model=model,
            val_loader=val_loader,
            criterion=criterion,
            device=device
        )
    
        if not math.isfinite(train_loss) or not math.isfinite(val_loss):
            print(
                f"Stopping early: non-finite loss detected | "
                f"Train Loss: {train_loss} | Val Loss: {val_loss}"
            )
            break
    
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_maes.append(val_mae)
        val_rmses.append(val_rmse)
    
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )
    
        # Checkpoint nach jeder Epoche speichern
        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "input_dim": input_dim,
            "batch_size": BATCH_SIZE,
        }
    
        checkpoint_path = MODEL_PATH / f"advanced_baseline_v2_NN_{MODEL_TAG}_epoch_{epoch + 1:03d}.pt"
        torch.save(checkpoint, checkpoint_path)
    
        print(f"Saved checkpoint: {checkpoint_path}")
    
        # Prüfen, ob sich das Modell verbessert hat
        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            epochs_without_improvement = 0
        
            best_checkpoint_path = MODEL_PATH / f"advanced_baseline_v2_NN_{MODEL_TAG}_best_model.pt"
            torch.save(checkpoint, best_checkpoint_path)
        
            print(f"Saved new best model: {best_checkpoint_path}")
    
        else:
            epochs_without_improvement += 1
        
            print(
                f"No improvement for {epochs_without_improvement}/"
                f"{PATIENCE} epochs"
            )
        
            if epochs_without_improvement >= PATIENCE:
                print(
                    f"Early stopping after {epoch + 1} epochs. "
                    f"Best Val Loss: {best_val_loss:.4f}"
                )
                break
    history_df = pl.DataFrame({
        "epoch": range(1, len(train_losses) + 1),
        "train_loss": train_losses,
        "val_loss": val_losses,
        "val_mae": val_maes,
        "val_rmse": val_rmses,
    })
    history_path = MODEL_PATH / f"advanced_baseline_v2_NN_{MODEL_TAG}_training_history.parquet"
    history_df.write_parquet(history_path)
    print(f"Saved training history: {history_path}")

    device = get_torch_device()

    model, checkpoint = load_model_from_checkpoint(
        model_class=TaxiDemandAdvancedNN,
        checkpoint_path=(
            Path(MODEL_PATH)
            / f"advanced_baseline_v2_NN_{MODEL_TAG}_best_model.pt"
        ),
        device=device,
    )
    val_eval_df = collect_predictions(
        model,
        val_loader,
        device=device,
        embedding_input=False,
    )

    val_metrics_df = compute_regression_metrics(val_eval_df)
    print(val_metrics_df)

    figure, bucket_eval_df = plot_regression_diagnostics(
        val_eval_df
    )

    plt.show()
    print(bucket_eval_df)
    return {
        "model_tag": MODEL_TAG,
        "model_path": MODEL_PATH / f"advanced_baseline_v2_NN_{MODEL_TAG}_best_model.pt",
        "history_path": history_path,
        "metrics": val_metrics_df,
        "bucket_metrics": bucket_eval_df,
    }


training_results = {}
for dataset_config in DATASET_CONFIGS:
    result = train_dataset_v2(dataset_config)
    training_results[result["model_tag"]] = result

print("Completed models:", list(training_results))


Epoch 1/100 | Train Loss: 0.4514 | Val Loss: 0.3081 | Val MAE: 4.1659 | Val RMSE: 18.3501
Saved checkpoint: ../models/advanced_baseline_v2_NN_epoch_001.pt
Saved new best model: ../models/advanced_baseline_v2_NN_best_model.pt
Epoch 2/100 | Train Loss: 0.2581 | Val Loss: 0.2277 | Val MAE: 3.2362 | Val RMSE: 13.6614
Saved checkpoint: ../models/advanced_baseline_v2_NN_epoch_002.pt
Saved new best model: ../models/advanced_baseline_v2_NN_best_model.pt
Epoch 3/100 | Train Loss: 0.2155 | Val Loss: 0.2106 | Val MAE: 2.9785 | Val RMSE: 12.6903
Saved checkpoint: ../models/advanced_baseline_v2_NN_epoch_003.pt
Saved new best model: ../models/advanced_baseline_v2_NN_best_model.pt
Epoch 4/100 | Train Loss: 0.2053 | Val Loss: 0.2030 | Val MAE: 2.7565 | Val RMSE: 11.1450
Saved checkpoint: ../models/advanced_baseline_v2_NN_epoch_004.pt
Saved new best model: ../models/advanced_baseline_v2_NN_best_model.pt
Epoch 5/100 | Train Loss: 0.2008 | Val Loss: 0.1987 | Val MAE: 2.8007 | Val RMSE: 11.5770
Saved chec

KeyboardInterrupt: 

### Load best model after training

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.

## Analyse model performance

In [ ]:
# Executed inside train_dataset_v2 for every dataset configuration.